In [2]:
pip install biopython primer3-py pandas


In [4]:
#WINDOWS CREATION

from Bio import SeqIO #Bio.AlignIO is a module in Biopython used for reading and writing MSA files

input_fasta = "consensus.fa"  #This is the consensus file of aligned COVID-19 file
window_size = 200
step = 80

record = SeqIO.read(input_fasta, "fasta")

windows = []
for start in range(0, len(record.seq) - window_size + 1, step):
    end = start + window_size
    window_seq = record.seq[start:end]

    windows.append({
        "id": f"window_{start+1}_{end}",
        "seq": window_seq
    })

# Write windows to FASTA
with open("windows.fa", "w") as out:
    for w in windows:
        out.write(f">{w['id']}\n{w['seq']}\n")

print(f"Generated {len(windows)} windows")


Generated 372 windows


In [15]:
from Bio import AlignIO

alignment_file = "229E.fa"
min_len = 17
allow_gaps = False   # set True ONLY if you really want gap-tolerant regions

alignment = AlignIO.read(alignment_file, "fasta")
seqs = [str(rec.seq) for rec in alignment]

num_seqs = len(seqs)
aln_len = len(seqs[0])

print(f"Sequences: {num_seqs}")
print(f"Alignment length: {aln_len}")

# ----------------------------------------
# STEP 1: Find conserved positions STRICTLY
# ----------------------------------------
strict_conserved = []

for i in range(aln_len):
    bases = {seq[i] for seq in seqs}

    if not allow_gaps and "-" in bases:
        strict_conserved.append(False)
    elif len(bases) == 1:
        strict_conserved.append(True)
    else:
        strict_conserved.append(False)

# ----------------------------------------
# STEP 2: Merge continuous conserved blocks
# ----------------------------------------
regions = []
start = None

for i, is_cons in enumerate(strict_conserved):
    if is_cons and start is None:
        start = i
    elif not is_cons and start is not None:
        end = i - 1
        if (end - start + 1) >= min_len:
            regions.append((start, end))
        start = None

# catch final region
if start is not None:
    end = aln_len - 1
    if (end - start + 1) >= min_len:
        regions.append((start, end))

# ----------------------------------------
# OUTPUT
# ----------------------------------------
print("\nHighly conserved continuous regions (≥17 bp):\n")

for idx, (start, end) in enumerate(regions, 1):
    seq = seqs[0][start:end+1]
    print(f"Region {idx}: {start+1}-{end+1} ({end-start+1} bp)")
    print(f"Sequence: {seq}\n")

Sequences: 106
Alignment length: 32247

Highly conserved continuous regions (≥17 bp):



#Prototype programme to exclude known primers, probes and continuous conserved stretches from other coronaviruses.

In [ ]:
#Prototype programme to exclude known primers, probes and continuous conserved stretches from other coronaviruses.

from Bio import SeqIO

# INPUT FILES
windows_fasta = "windows.fa"     # candidate windows
exclude_file = "exclude.txt"     # existng primers & other-CoV conserved regions

output_fasta = "filtered_windows.fa"

# LOAD EXCLUSION SEQUENCES
exclude_seqs = set()

with open(exclude_file) as f:
    for line in f:
        seq = line.strip().upper()
        if seq:
            exclude_seqs.add(seq)

print(f"Loaded {len(exclude_seqs)} exclusion sequences")

# FILTER WINDOWS
kept = []
removed = []

for record in SeqIO.parse(windows_fasta, "fasta"):
    window_seq = str(record.seq).upper()

    exclude_hit = False
    for ex in exclude_seqs:
        if ex in window_seq:
            exclude_hit = True
            break

    if exclude_hit:
        removed.append(record)
    else:
        kept.append(record)

# WRITE OUTPUT
SeqIO.write(kept, output_fasta, "fasta")

# REPORT
print(f"Total windows      : {len(kept) + len(removed)}")
print(f"Excluded windows   : {len(removed)}")
print(f"Retained candidates: {len(kept)}")
print(f"\nFiltered windows written to: {output_fasta}")


ModuleNotFoundError: No module named 'Bio'

In [ ]:
import pandas as pd
import zipfile
import io

zip_path = "IDTPrimerQuest.zip"

fasta_lines = []

with zipfile.ZipFile(zip_path, 'r') as z:
    for file in z.namelist():

        if (
            file.endswith(".xlsx")
            and not file.startswith("__MACOSX")
            and not file.split("/")[-1].startswith("._")
        ):
            df = pd.read_excel(io.BytesIO(z.read(file)), engine="openpyxl")

            # extract window name
            window_name = file.split("/")[-1].replace(".xlsx", "")

            # keep only oligos
            df = df[df["Type"].isin(["Forward Primer", "Reverse Primer", "Probe"])]

            # 🔥 group by AssaySet
            grouped = df.groupby("AssaySet")

            for set_id, group in grouped:

                for _, row in group.iterrows():
                    type_clean = row["Type"].replace(" ", "")
                    seq = str(row["Sequence"]).strip()

                    name = f"{window_name}_{type_clean}_{set_id}"

                    fasta_lines.append(f">{name}")
                    fasta_lines.append(seq)

with open("all_oligos.fasta", "w") as f:
    f.write("\n".join(fasta_lines))

print("Proper FASTA created")

In [ ]:
with open("all_oligos.fasta", "r") as f:
    text = f.read()

text = text.replace(">>", ">")

with open("all_oligos_fixed.fasta", "w") as f:
    f.write(text)

print("Fixed FASTA created")

In [ ]:
# =========================
# HETERODIMER CHECK (FINAL VERSION)
# =========================

from Bio import SeqIO
import primer3

# -------------------------
# SETTINGS
# -------------------------
INPUT_FASTA = "all_oligos_fixed.fasta"
OUTPUT_FILE = "heterodimer_results.csv"

DG_THRESHOLD = -9   # strong dimers only (important fix)
TM_THRESHOLD = 40   # optional but recommended
PROGRESS_EVERY = 1000

# -------------------------
# LOAD SEQUENCES
# -------------------------
records = list(SeqIO.parse(INPUT_FASTA, "fasta"))
oligos = [(rec.id, str(rec.seq)) for rec in records]

n = len(oligos)
print(f"Loaded {n} oligos")

# -------------------------
# WRITE HEADER
# -------------------------
with open(OUTPUT_FILE, "w") as f:
    f.write("Oligo1,Oligo2,Tm,dG\n")

# -------------------------
# MAIN LOOP (OPTIMIZED)
# -------------------------
count = 0
hits = 0

for i in range(n):
    name1, seq1 = oligos[i]

    for j in range(i + 1, n):   # avoids duplicates + self comparison
        name2, seq2 = oligos[j]

        count += 1

        # progress update
        if count % PROGRESS_EVERY == 0:
            print(f"Processed {count} pairs | Hits: {hits}")

        try:
            # ✅ updated (non-deprecated function)
            result = primer3.calc_heterodimer(seq1, seq2)

            # ✅ correct filtering logic
            if result.dg < DG_THRESHOLD and result.tm > TM_THRESHOLD:
                hits += 1

                # ✅ memory-safe write
                with open(OUTPUT_FILE, "a") as f:
                    f.write(f"{name1},{name2},{result.tm},{result.dg}\n")

        except Exception as e:
            print(f"Error with {name1} vs {name2}: {e}")

# -------------------------
# SUMMARY
# -------------------------
print("\nDone!")
print(f"Total pairs checked: {count}")
print(f"Strong dimers found: {hits}")